In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-2505060022-WildanMaulana") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("SparkSession siap. Versi Spark:", spark.version)

26/11/21 12:27:12 WARN Utils: Your hostname, xicrth resolves to a loopback address: 127.0.1.1; using 192.168.110.164 instead (on interface wlp62s0)
26/11/21 12:27:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/21 12:27:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [3]:
# Membaca tugas12_ulasan.csv menggunakan Spark
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Distribusi label (Spark):")
df.groupBy("ulasan_positif").count().orderBy("ulasan_positif").show()

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

Distribusi 

+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             0| 1610|
|             1| 1390|
+--------------+-----+



In [4]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Tahap 4: OneHotEncoder untuk ketiga kolom indeks
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# Tahap 5: VectorAssembler menggabungkan fitur numerik & hasil one-hot encoding
assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "nama_menu_vec", "cabang_vec", "waktu_pesan_vec"],
    outputCol="features"
)

# Tahap 6: LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Rangkai menjadi satu Pipeline
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression berhasil didefinisikan dengan", len(pipeline_lr.getStages()), "tahap.")

Pipeline Logistic Regression berhasil didefinisikan dengan 6 tahap.


In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split data 80:20 dengan seed=77
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Jumlah data training: {train.count()} | Jumlah data testing: {test.count()}")

# Latih pipeline Logistic Regression
model_lr = pipeline_lr.fit(train)
predictions_lr = model_lr.transform(test)

# Evaluator
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(predictions_lr)
f1_lr = ev_f1.evaluate(predictions_lr)
auc_lr = ev_auc.evaluate(predictions_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix (Logistic Regression):")
predictions_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Jumlah data training: 2411 | Jumlah data testing: 589


=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix (Logistic Regression):


+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [6]:
from pyspark.ml.classification import RandomForestClassifier

# Tahap klasifikasi diganti RandomForestClassifier dengan numTrees=30
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)

# Pipeline baru dengan 5 tahap awal yang identik
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(predictions_rf)
f1_rf = ev_f1.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Evaluasi Random Forest (numTrees=30) ===")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

print("\nConfusion Matrix (Random Forest):")
predictions_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

print("=== Perbandingan Kedua Model ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== Evaluasi Random Forest (numTrees=30) ===
Accuracy : 0.7946
F1-score : 0.7943
AUC-ROC  : 0.8555

Confusion Matrix (Random Forest):
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  258|
|             0|       1.0|   56|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+

=== Perbandingan Kedua Model ===
Logistic Regression -> Accuracy: 0.7929 | F1: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7946 | F1: 0.7943 | AUC-ROC: 0.8555


In [7]:
# Pilih model terbaik berdasarkan AUC-ROC tertinggi
if auc_lr >= auc_rf:
    best_name = "Logistic Regression"
    best_model = model_lr
    best_preds = predictions_lr
    best_auc = auc_lr
else:
    best_name = "Random Forest"
    best_model = model_rf
    best_preds = predictions_rf
    best_auc = auc_rf

print(f"Model terbaik berdasarkan AUC-ROC: {best_name} (AUC = {best_auc:.4f})")

# Filter baris yang salah diprediksi (label != prediction)
errors = best_preds.filter(col("ulasan_positif") != col("prediction"))
print(f"Total kesalahan prediksi: {errors.count()} dari {best_preds.count()} data testing")

print("\n10 Baris Kesalahan Prediksi:")
errors.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10, truncate=False)

print("Statistik waktu_tunggu_menit pada baris yang SALAH diprediksi:")
errors.select("waktu_tunggu_menit", "qty").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

print("Distribusi waktu_tunggu_menit pada kesalahan (top 10 terbanyak):")
errors.groupBy("waktu_tunggu_menit").count().orderBy(col("count").desc()).show(10)

Model terbaik berdasarkan AUC-ROC: Logistic Regression (AUC = 0.8650)


Total kesalahan prediksi: 122 dari 589 data testing

10 Baris Kesalahan Prediksi:
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  |5                 |0 

+-------+------------------+------------------+
|summary|waktu_tunggu_menit|               qty|
+-------+------------------+------------------+
|  count|               122|               122|
|   mean|12.647540983606557| 2.942622950819672|
| stddev| 5.008954182505435|1.3983412861411042|
|    min|                 2|                 1|
|    25%|                 9|                 2|
|    50%|                12|                 3|
|    75%|                16|                 4|
|    max|                24|                 5|
+-------+------------------+------------------+

Distribusi waktu_tunggu_menit pada kesalahan (top 10 terbanyak):
+------------------+-----+
|waktu_tunggu_menit|count|
+------------------+-----+
|                14|   13|
|                11|   11|
|                13|   10|
|                12|   10|
|                16|    8|
|                10|    8|
|                 8|    7|
|                15|    6|
|                 6|    6|
|                17|    6|
+------

In [8]:
# Melihat pengaruh fitur pada model Logistic Regression dan Random Forest
feature_meta = model_lr.stages[4].getInputCols()
lr_stage = model_lr.stages[-1]
rf_stage = model_rf.stages[-1]

attrs = predictions_lr.schema["features"].metadata["ml_attr"]["attrs"]
feat_names = []
for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in attrs:
        for a in attrs[attr_type]:
            feat_names.append((a["idx"], a["name"]))
feat_names.sort(key=lambda x: x[0])
names = [x[1] for x in feat_names]

print("=== Koefisien Logistic Regression ===")
for name, coef in zip(names, lr_stage.coefficients.toArray()):
    print(f"{name:35s} : {coef:+.4f}")
print(f"{'Intercept':35s} : {lr_stage.intercept:+.4f}")

print("\n=== Feature Importances Random Forest ===")
for name, imp in sorted(zip(names, rf_stage.featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{name:35s} : {imp:.4f}")

=== Koefisien Logistic Regression ===
qty                                 : -0.0108
waktu_tunggu_menit                  : -0.2220
nama_menu_vec_Kopi Susu             : +0.1171
nama_menu_vec_Cappuccino            : +0.0194
nama_menu_vec_Americano             : -0.0855
nama_menu_vec_Roti Bakar            : -0.1443
nama_menu_vec_Es Teh                : -0.0191
cabang_vec_Yogyakarta               : +0.0645
cabang_vec_Semarang                 : -0.1140
waktu_pesan_vec_Pagi                : -0.0651
waktu_pesan_vec_Malam               : +0.2530
waktu_pesan_vec_Sore                : +0.1857
Intercept                           : +2.6615

=== Feature Importances Random Forest ===
waktu_tunggu_menit                  : 0.9475
qty                                 : 0.0137
cabang_vec_Semarang                 : 0.0074
waktu_pesan_vec_Malam               : 0.0053
nama_menu_vec_Kopi Susu             : 0.0049
nama_menu_vec_Roti Bakar            : 0.0048
cabang_vec_Yogyakarta               : 0.0044
waktu_

---
## E. Kesimpulan Bisnis

Berdasarkan hasil pemodelan klasifikasi pada data pesanan **Kopi Nusantara**, fitur yang paling berpengaruh kuat terhadap ulasan positif pelanggan adalah **`waktu_tunggu_menit`**. Pada model Logistic Regression, fitur ini memiliki koefisien negatif paling besar (`-0.2220`), sedangkan pada Random Forest fitur ini mendominasi dengan *feature importance* sebesar **`0.9475` (94,75%)**—jauh melampaui `qty` (`0.0137`), `cabang`, `nama_menu`, maupun `waktu_pesan`.

Sejalan dengan analisis kesalahan pada Bagian D yang menunjukkan zona kritis berada di sekitar **11–14 menit** (median kesalahan **12 menit**), manajemen Kopi Nusantara disarankan untuk mengambil langkah konkret berikut:
1. **Menetapkan SLA Waktu Tunggu Maksimal di Bawah 10 Menit:** Jadikan batas 10 menit sebagai target operasional utama di dapur dan bar, karena melewati ambang 12 menit peluang pelanggan memberikan ulasan positif turun drastis.
2. **Optimalisasi Alur Persiapan Menu & Antrean Jam Sibuk:** Terapkan sistem persiapan bahan setengah jadi (*pre-batching*) untuk menu populer serta evaluasi alokasi barista/staf di cabang dengan performa lebih rendah (misalnya cabang Semarang yang memiliki koefisien negatif `-0.1140` pada model).
3. **Kompensasi Proaktif Saat Terjadi Keterlambatan:** Jika sistem kasir mendeteksi estimasi waktu tunggu melewati 12 menit, staf dapat memberikan pemberitahuan dini atau kompensasi kecil (seperti kupon diskon kunjungan berikutnya) untuk menjaga kepuasan pelanggan dan mencegah ulasan negatif.

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
